# 🎓 Interactive Lecture Intelligence - Phase 2 ASR on Colab

**Use this notebook to run the heavy ASR (speech-to-text) processing on free GPU.**

⏱️ Time: 10-30 minutes with T4 GPU (vs 2-8 hours on CPU)

---

## Step 1: Enable GPU Runtime

1. Click **Runtime** → **Change runtime type**
2. Select **T4 GPU** (free tier)
3. Click **Save**

In [ ]:
# Verify GPU is available
!nvidia-smi

---

## Step 2: Clone Repository

In [ ]:
# Clone your repository
!git clone https://github.com/YOUR_USERNAME/speech_rag.git
%cd speech_rag

---

## Step 3: Install Dependencies

In [ ]:
# Install Python packages
!pip install -q openai-whisper torch torchaudio librosa soundfile pydub PyYAML

---

## Step 4: Upload Audio Files

**Option A: Upload from your computer**

In [ ]:
from google.colab import files
import os

# Create directory
os.makedirs('data/raw_audio', exist_ok=True)

# Upload files (click button to select)
print("📤 Click 'Choose Files' to upload your lecture audio...")
uploaded = files.upload()

# Move to correct directory
for filename in uploaded.keys():
    !mv "{filename}" data/raw_audio/
    print(f"✅ Uploaded: {filename}")

# List uploaded files
!ls -lh data/raw_audio/

**Option B: Mount Google Drive** (if files are in Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy from Drive to working directory
!cp /content/drive/MyDrive/lectures/*.mp3 data/raw_audio/
!ls -lh data/raw_audio/

---

## Step 5: Configure for GPU

In [ ]:
# Verify config uses GPU
!cat config/config.yaml | grep -A 5 "Phase 2"

In [ ]:
# Update config to ensure GPU is used
import yaml

with open('config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Ensure GPU settings
config['device'] = 'cuda'
config['asr_model_size'] = 'large-v3'  # Best quality
config['batch_size'] = 8  # GPU can handle larger batches

with open('config/config.yaml', 'w') as f:
    yaml.dump(config, f)

print("✅ Config updated for GPU processing")

---

## Step 6: Run ASR (Speech-to-Text) 🚀

This is the heavy computation phase. With T4 GPU:
- ~5-10 min per hour of audio
- Model downloads once (~3 GB)

In [ ]:
# Run Phase 2 ASR pipeline
!python scripts/run_phase2_asr.py

print("\n" + "="*60)
print("✅ ASR COMPLETE!")
print("="*60)

---

## Step 7: Verify Transcripts

In [ ]:
# List generated transcripts
!ls -lh data/transcripts/

# Preview first transcript
import json
import glob

transcript_files = glob.glob('data/transcripts/*.json')
if transcript_files:
    with open(transcript_files[0], 'r') as f:
        data = json.load(f)
    
    print(f"\n📄 Sample: {transcript_files[0]}")
    print(f"Lecture ID: {data.get('lecture_id', 'N/A')}")
    print(f"Duration: {data.get('duration', 0):.1f}s")
    print(f"\nFirst 500 characters:")
    print(data.get('text', '')[:500] + "...")
else:
    print("⚠️ No transcripts found!")

---

## Step 8: Download Results

Download transcripts to continue locally

In [ ]:
# Zip transcripts and segments
!zip -r transcripts_output.zip data/transcripts/ data/segments/

# Download
from google.colab import files
files.download('transcripts_output.zip')

print("\n📦 Download started!")
print("\nNext steps on your local machine:")
print("  1. Extract transcripts_output.zip to your project directory")
print("  2. Run: python scripts/run_phase3_indexing.py")
print("  3. Run: python scripts/generate_flashcards.py")
print("  4. Run: python scripts/generate_summaries.py")
print("  5. Launch: streamlit run frontend/streamlit_app.py")

---

## ⚡ Quick Commands Reference

```bash
# Check GPU
!nvidia-smi

# List audio files
!ls -lh data/raw_audio/

# Process single file
!python scripts/run_phase2_asr.py --audio data/raw_audio/lecture_01.mp3

# Check logs
!tail -50 logs/asr_pipeline.log

# Disk usage
!du -sh data/
```

---

## 🔧 Troubleshooting

### Out of Memory
```python
# Reduce batch size in config
config['batch_size'] = 4  # or even 2
```

### Slow Processing
- Verify GPU: `!nvidia-smi`
- Check device: `config['device']` should be `'cuda'`

### Audio Format Issues
```bash
# Convert to supported format
!ffmpeg -i input.m4a -ar 16000 -ac 1 output.wav
```